In [3]:
### Imports

import numpy as np
import gurobipy as gb
import csv
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator



### Instance formulation

We create 3 functions that return a parametre dictionary for all 3 instances

In [4]:
def small_instance():
    # time horizon for the deployment
    T = 25

    # Description of nodes
    # Format: node_id, node_type_id, coord_x, coord_y, E (if relevant)
    #	* each row corresponds to 1 site
    #	* node_id = index of the node
    #   * node_type_id {0: capture site, 1: transfer site, 2: storage site}
    #	* (coord_x, coord_y) = geographical coordinates of the site
    #	* E = yearly emissions of this site (without any carbon capture technology)

    # Array of nodes
    nodes = np.array([
        [0, 2, 0, 10, 0], 
        [1, 0, 9, 10, 200],
        [2, 0, 5, 7, 250],
        [3, 0, 10, 4, 400],
        [4, 0, 5, 2, 200],
        [5, 0, 0, 0, 350]
    ])

    N = [int(node[0]) for node in nodes]

    Nc_nodes = [node for node in nodes if int(node[1]) == 0]
    Nt_nodes = [node for node in nodes if int(node[1]) == 1]
    Ns_nodes = [node for node in nodes if int(node[1]) == 2]

    Nc = [int(node[0]) for node in Nc_nodes]
    Nt = [int(node[0]) for node in Nt_nodes]
    Ns = [int(node[0]) for node in Ns_nodes]

    Lij = np.zeros((len(nodes), len(nodes)), dtype="float")
    M1 = 0
    E = {}
    b = {}
    for node_i in nodes:
        if node_i[1] == 0:
            M1 += node_i[4]  # The big M is used to allow the flow of emissions so taking the sum of all the emissions is a reasonable big M variable
            E[node_i[0]] = node_i[4]
        if node_i[1] == 2:
            b[node_i[0]] = node_i[4]
        for node_j in nodes:
            Lij[node_i[0], node_j[0]] = np.sqrt((node_i[2] - node_j[2])**2 + (node_i[3] - node_j[3])**2)


    M2 = M1 # The sum is also reasonable for the second big M constraint

    T = np.array([t for t in range(T)])

    # -- array (of size N*N) where element w_ij indicates whether or not a rail connection exists between two nodes
    #	* w_ij = 1 if a rail connection exists between node i (row index) and j (column index), w_ij = 0 otherwise

    # Existing rail connections
    w_rail = np.array([
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
    ], dtype=int)

    # beta[mode][0]: cost per unit transported per unit distance
    # beta[mode][1]: additional cost per unit transported
    beta = {
        "truck": np.array([1_000_000, 0]),
        "rail": np.array([1, 0.5]),
        "pipe": np.array([0.05, 0]),
    }

    gamma = {"truck": 0, "rail": 0.003, "pipe": 0}
    alpha = np.array([0.3, 27])

    # Assumed capture efficiency: replace with your actual value
    emission_coefficient = 0.90

    total_emissions = sum(E.values())
    max_captured = emission_coefficient * total_emissions

    # Flow bounds; sufficient for routing captured carbon without cycles
    M1 = max_captured
    M2 = max_captured

    # Bounds on transport emissions MINUS captured emissions.
    # Assumption: progress from no required reduction to a 90% reduction.
    Q_init = 0.0
    Qt = -0.90 * total_emissions

    params = {
        "T": T,
        "N": N,
        "nodes": nodes,
        "w_rail": w_rail,
        "beta": beta,
        "gamma": gamma,
        "alpha": alpha,
        "Nc_nodes": Nc_nodes,
        "Nt_nodes": Nt_nodes,
        "Ns_nodes": Ns_nodes,
        "Nc": Nc,
        "Nt": Nt,
        "Ns": Ns,
        "Lij": Lij,
        "M1": M1,
        "M2": M2,
        "E": E,
        "b": b,
        "emission_coefficient": emission_coefficient,
        "Q_init": Q_init,
        "Qt": Qt,
    }

    return params


def medium_instance():
    T = 25
    # T = 40

    # node_id, node_type_id, coord_x, coord_y, E or date_available
    # Types: 0 = capture, 1 = transfer, 2 = storage
    nodes = np.array([
        [0,  2, 0,   10, 0],
        [1,  0, 9,   10, 200],
        [2,  0, 5,   7,  250],
        [3,  0, 10,  4,  400],
        [4,  0, 5,   2,  200],
        [5,  0, 0,   0,  350],
        [6,  0, 4.5, 1,  50],
        [7,  2, 6,   4,  14],
        [8,  0, 8.5, 0,  65],
        [9,  0, 2,   6,  90],
        [10, 0, 7,   8,  45],
    ])

    N = [int(node[0]) for node in nodes]

    Nc_nodes = [node for node in nodes if int(node[1]) == 0]
    Nt_nodes = [node for node in nodes if int(node[1]) == 1]
    Ns_nodes = [node for node in nodes if int(node[1]) == 2]

    Nc = [int(node[0]) for node in Nc_nodes]
    Nt = [int(node[0]) for node in Nt_nodes]
    Ns = [int(node[0]) for node in Ns_nodes]

    Lij = np.zeros((len(nodes), len(nodes)), dtype=float)
    E = {}
    b = {}

    for node_i in nodes:
        i = int(node_i[0])

        if int(node_i[1]) == 0:
            E[i] = node_i[4]
        if int(node_i[1]) == 2:
            b[i] = int(node_i[4])

        for node_j in nodes:
            j = int(node_j[0])
            Lij[i, j] = np.sqrt(
                (node_i[2] - node_j[2])**2
                + (node_i[3] - node_j[3])**2
            )

    T = np.arange(T)

    # Existing rail connections
    w_rail = np.array([
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    ], dtype=int)

    beta = {
        "truck": np.array([5, 44]),
        "rail": np.array([1.6, 31]),
        "pipe": np.array([0.08, 0]),
    }

    gamma = {"truck": 0.0007, "rail": 0.0001, "pipe": 0}
    alpha = np.array([0.0005, 3000])

    # Same assumptions as small_instance()
    emission_coefficient = 0.90
    total_emissions = sum(E.values())
    max_captured = emission_coefficient * total_emissions

    M1 = max_captured
    M2 = max_captured

    Q_init = 0.0
    Qt = -0.90 * total_emissions

    params = {
        "T": T,
        "N": N,
        "nodes": nodes,
        "w_rail": w_rail,
        "beta": beta,
        "gamma": gamma,
        "alpha": alpha,
        "Nc_nodes": Nc_nodes,
        "Nt_nodes": Nt_nodes,
        "Ns_nodes": Ns_nodes,
        "Nc": Nc,
        "Nt": Nt,
        "Ns": Ns,
        "Lij": Lij,
        "M1": M1,
        "M2": M2,
        "E": E,
        "b": b,
        "emission_coefficient": emission_coefficient,
        "Q_init": Q_init,
        "Qt": Qt,
    }

    return params


def large_instance():
    T = 25

    # node_id, node_type_id, coord_x, coord_y, E or date_available
    # Types: 0 = capture, 1 = transfer, 2 = storage
    nodes = np.array([
        [0,  2, 0,   10, 0],
        [1,  0, 9,   10, 200],
        [2,  0, 5,   7,  250],
        [3,  0, 10,  4,  400],
        [4,  0, 5,   2,  200],
        [5,  0, 0,   0,  350],
        [6,  0, 4.5, 1,  50],
        [7,  2, 6,   4,  14],
        [8,  0, 8.5, 0,  65],
        [9,  0, 2,   6,  90],
        [10, 0, 7,   8,  45],
        [11, 1, 1,   10, 0],
        [12, 0, 4,   7,  320],
        [13, 1, 2,   2,  0],
        [14, 0, 3,   4,  80],
        [15, 0, 6,   8,  150],
        [16, 0, 2,   1,  60],
        [17, 0, 3,   2,  80],
    ])

    N = [int(node[0]) for node in nodes]

    Nc_nodes = [node for node in nodes if int(node[1]) == 0]
    Nt_nodes = [node for node in nodes if int(node[1]) == 1]
    Ns_nodes = [node for node in nodes if int(node[1]) == 2]

    Nc = [int(node[0]) for node in Nc_nodes]
    Nt = [int(node[0]) for node in Nt_nodes]
    Ns = [int(node[0]) for node in Ns_nodes]

    Lij = np.zeros((len(nodes), len(nodes)), dtype=float)
    E = {}
    b = {}

    for node_i in nodes:
        i = int(node_i[0])

        if int(node_i[1]) == 0:
            E[i] = node_i[4]
        if int(node_i[1]) == 2:
            b[i] = int(node_i[4])

        for node_j in nodes:
            j = int(node_j[0])
            Lij[i, j] = np.sqrt(
                (node_i[2] - node_j[2])**2
                + (node_i[3] - node_j[3])**2
            )

    T = np.arange(T)

    # Existing rail connections (all other entries are zero)
    w_rail = np.zeros((len(nodes), len(nodes)), dtype=int)

    for i, j in [(2, 3), (2, 11), (11, 13)]:
        w_rail[i, j] = 1
        w_rail[j, i] = 1

    beta = {
        "truck": np.array([5, 44]),
        "rail": np.array([1.6, 31]),
        "pipe": np.array([0.08, 0]),
    }

    gamma = {"truck": 0.0007, "rail": 0.0001, "pipe": 0}
    alpha = np.array([0.0005, 3000])

    # Same assumptions as small_instance()
    emission_coefficient = 0.90
    total_emissions = sum(E.values())
    max_captured = emission_coefficient * total_emissions

    M1 = max_captured
    M2 = max_captured

    Q_init = 0.0
    Qt = -0.90 * total_emissions

    params = {
        "T": T,
        "N": N,
        "nodes": nodes,
        "w_rail": w_rail,
        "beta": beta,
        "gamma": gamma,
        "alpha": alpha,
        "Nc_nodes": Nc_nodes,
        "Nt_nodes": Nt_nodes,
        "Ns_nodes": Ns_nodes,
        "Nc": Nc,
        "Nt": Nt,
        "Ns": Ns,
        "Lij": Lij,
        "M1": M1,
        "M2": M2,
        "E": E,
        "b": b,
        "emission_coefficient": emission_coefficient,
        "Q_init": Q_init,
        "Qt": Qt,
    }

    return params



### Model Formulation

In order to represent the model we create a class for the model where the varaibles are set, the objective function is defined and all the constraints are created

In [5]:
class CarbonCaptureConstraints:
    def __init__(self, m, params):
        self.m = m
        self.params_ = params

    def define_variables(self):
        T, N= (
            self.params_["T"],
            self.params_["N"],
        )

        F_rail = self.m.addVars(N, N, T, vtype=gb.GRB.CONTINUOUS, lb=0)

        w_pipe = self.m.addVars(N, N, T, vtype=gb.GRB.BINARY, name="w_ij_pipe")
        S_pipe = self.m.addVars(N, N, T,  vtype=gb.GRB.CONTINUOUS, lb=0)
        F_pipe = self.m.addVars(N, N, T, vtype=gb.GRB.CONTINUOUS, lb=0)

        F_truck = self.m.addVars(N, N, T, vtype=gb.GRB.CONTINUOUS, lb=0)

        c_site = self.m.addVars(N, T, vtype=gb.GRB.BINARY)

        self.vars_ = {"F_rail": F_rail, "w_pipe": w_pipe, "S_pipe": S_pipe, "F_pipe": F_pipe, "F_truck": F_truck, "c_site": c_site}

        return self.vars_

    def function_objective(self):
        T, N, nodes, w_rail, beta, gamma, alpha, Nc_nodes, Nt_nodes, Ns_nodes, Nc, Nt, Ns, Lij = (
            self.params_["T"],
            self.params_["N"],
            self.params_["nodes"],
            self.params_["w_rail"],
            self.params_["beta"],
            self.params_["gamma"],
            self.params_["alpha"],
            self.params_["Nc_nodes"],
            self.params_["Nt_nodes"],
            self.params_["Ns_nodes"],
            self.params_["Nc"],
            self.params_["Nt"],
            self.params_["Ns"],
            self.params_["Lij"]
        )

        S_pipe, w_pipe = self.vars_["S_pipe"], self.vars_["w_pipe"]
        F_pipe = self.vars_["F_pipe"]
        F_rail = self.vars_["F_rail"]
        F_truck = self.vars_["F_truck"]
        c_site = self.vars_["c_site"]

        f1 = gb.quicksum(
            beta["truck"][0] * F_truck[i,j,t] * Lij[i,j] + beta["truck"][1] * F_truck[i,j,t] for i in N for j in N for t in T
        )

        f2 = gb.quicksum(
            beta["rail"][0] * F_rail[i,j,t] * Lij[i,j] + beta["rail"][1] * F_rail[i,j,t] for i in N for j in N for t in T
        )

        f3 = gb.quicksum(
            beta["pipe"][0] * F_pipe[i,j,t] * Lij[i,j] + beta["pipe"][1] * F_pipe[i,j,t] for i in N for j in N for t in T
        )

        f4 = gb.quicksum(
            alpha[0] * S_pipe[i,j,t] + alpha[1] * Lij[i,j] * w_pipe[i,j,t] for i in N for j in N if i<j for t in T
        )

        self.m.setObjective(f1 + f2 + f3 + f4, gb.GRB.MINIMIZE)
        
    def pipeline_constraints(self):
        S_pipe = self.vars_["S_pipe"]
        w_pipe = self.vars_["w_pipe"]
        F_pipe = self.vars_["F_pipe"]
        
        T, N, M1 = (
            self.params_["T"],
            self.params_["N"],
            self.params_["M1"]
        )

        for i in N:
            for j in N:
                for t in T:
                    self.m.addConstr(
                        S_pipe[i,j,t] - S_pipe[j,i,t] == 0, name=f"Pipeline Capacity constraint S_{i},{j},{t} = S_{j},{i},{t}"
                    )

        for i in N:
            for j in N:
                for t in T:
                    if t < max(T):
                        self.m.addConstr(
                            S_pipe[i,j,t+1] - S_pipe[i,j,t] >= 0, name="Non-decreasing pipeline capacity over time"
                        )

        for i in N:
            for j in N:
                for t in T:
                    if t < max(T):
                        self.m.addConstr(
                            w_pipe[i,j,t+1] - w_pipe[i,j,t] >= 0, name="Pipeline doesn't disappear after being created"
                        )

        for i in N:
            for j in N:
                for t in T:
                    if t < max(T):
                        self.m.addConstr(
                            S_pipe[i, j, t + 1] - S_pipe[i, j, t]
                            <= M1 * (
                                w_pipe[i, j, t + 1] - w_pipe[i, j, t]
                            ),
                            name=f"Capacity_fixed_after_installation_{i}_{j}_{t}"
                        )

        for i in N:
            for j in N:
                if i<j:
                    for t in T:
                        self.m.addConstr(
                            F_pipe[i,j,t] + F_pipe[j,i,t] <= S_pipe[i,j,t], name="The total flux in a pipeline cannot be larger than it's capacity"
                        )

        for i in N:
            for j in N:
                if i<j:
                    for t in T:
                        self.m.addConstr(
                            w_pipe[i,j,t] - w_pipe[j,i,t] == 0, name="A pipeline is bidirectional"
                        )

        for i in N:
            for t in T:
                self.m.addConstr(
                    w_pipe[i,i,t] == 0, name="A pipeline doesn't loop around to the same node"
                )
                self.m.addConstr(
                    F_pipe[i,i,t] == 0, name="A pipeline flow doesn't loop around to the same node"
                )

        for i in N:
            for j in N:
                for t in T:
                    self.m.addConstr(
                        S_pipe[i,j,t] <= M1 * w_pipe[i,j,t]
                    )

    def rail_constraints(self):
        F_rail = self.vars_["F_rail"]

        T, N, w_rail, M1= (
            self.params_["T"],
            self.params_["N"],
            self.params_["w_rail"],
            self.params_["M1"]
        )

        for i in N:
            for j in N:
                for t in T:
                    self.m.addConstr(
                        F_rail[i,j,t] <= w_rail[i,j] * M1, name="There is only emissions sent through rail if a line exists"
                    )


        for i in N:
            for t in T:
                self.m.addConstr(self.vars_["F_truck"][i, i, t] == 0, name="No truck self loops")
                self.m.addConstr(self.vars_["F_rail"][i, i, t] == 0, name="No rail self loops")

    def flow_constraints(self):
        F_pipe = self.vars_["F_pipe"]
        F_rail = self.vars_["F_rail"]
        F_truck = self.vars_["F_truck"]
        c_site = self.vars_["c_site"]

        T, N, Nc, Nt, Ns, emission_coefficient, E, M2, b = (
            self.params_["T"],
            self.params_["N"],
            self.params_["Nc"],
            self.params_["Nt"],
            self.params_["Ns"], 
            self.params_["emission_coefficient"],
            self.params_["E"], 
            self.params_["M2"],
            self.params_["b"]
        )
    
        for i in Nc:
            for t in T:
                self.m.addConstr(
                    gb.quicksum(F_truck[j,i,t] + F_rail[j,i,t] + F_pipe[j,i,t] for j in N) - 
                    gb.quicksum(F_truck[i,j,t] + F_rail[i,j,t] + F_pipe[i,j,t] for j in N) ==
                    - emission_coefficient * E[i] * c_site[i,t],
                    name="The flow out of a capture site is equal to the amount of carbon captured at this site"
                )

        for i in Nt:
            for t in T:
                self.m.addConstr(
                    gb.quicksum(F_truck[j,i,t] + F_rail[j,i,t] + F_pipe[j,i,t] for j in N) - 
                    gb.quicksum(F_truck[i,j,t] + F_rail[i,j,t] + F_pipe[i,j,t] for j in N) == 0 ,
                    name="The flow of a transfer site is equal to zero"
                )

        for i in Ns:
            for t in T:
                active_site = int(t >= b[i])
                self.m.addConstr(
                    gb.quicksum(F_truck[j,i,t] + F_rail[j,i,t] + F_pipe[j,i,t] for j in N) <= M2 * active_site ,
                    name=f"Storage site {i} is available or not"
                )

        for i in Ns:
            for t in T:
                self.m.addConstr(
                    gb.quicksum(F_truck[i,j,t] + F_rail[i,j,t] + F_pipe[i,j,t] for j in N) == 0 ,
                    name=f"The flow out of storage site {i} is equal to zero"
                )

    def emission_management_constraints(self):
        T, N, gamma, Nc, emission_coefficient, E, Qt, Q_init, Lij = (
            self.params_["T"],
            self.params_["N"],
            self.params_["gamma"],
            self.params_["Nc"],
            self.params_["emission_coefficient"],
            self.params_["E"],
            self.params_["Qt"],
            self.params_["Q_init"],
            self.params_["Lij"]
        )
        
        F_pipe = self.vars_["F_pipe"]
        F_rail = self.vars_["F_rail"]
        F_truck = self.vars_["F_truck"]
        c_site = self.vars_["c_site"]
    
        for t in T:
            self.m.addConstr(
                gb.quicksum(
                    gamma["truck"] * F_truck[i,j,t] * Lij[i,j] + 
                    gamma["rail"] * F_rail[i,j,t] * Lij[i,j] + 
                    gamma["pipe"] * F_pipe[i,j,t] * Lij[i,j]
                    for i in N for j in N
                ) - gb.quicksum(emission_coefficient * E[i] * c_site[i,t] for i in Nc) 
                <= Q_init + (Qt - Q_init) * (t/max(T)),
                name="Emissions are reduced in a linear fashion in order to reach the emissions goal"
            )


class OptiModel:
    def __init__(self, model_name, TIME_LIMIT, model_instance):
        self.model_name = model_name
        self.TIME_LIMIT = TIME_LIMIT
        # self.params = small_instance()
        self.params = model_instance()
        # self.params = large_instance()
        self.vars = None
        self.model = None

    def build(self):
        m = gb.Model(self.model_name)
        m.setParam("TimeLimit", self.TIME_LIMIT)

        constraint_builder = CarbonCaptureConstraints(m=m, params=self.params)

        vars_ = constraint_builder.define_variables()

        constraint_builder.function_objective()

        constraint_builder.pipeline_constraints()
        constraint_builder.rail_constraints()
        constraint_builder.flow_constraints()
        constraint_builder.emission_management_constraints()

        m.update()
        self.model, self.vars = m, vars_
        return self.model, self.vars


### Code verification

Here we have a code that prints all the constraints into one txt file and their values in order to be able to see if there are any problems. 

In [6]:
def export_constraints_txt(model, filename="model_constraints.txt"):
    model.update()

    txt_path = Path(filename)
    lp_path = txt_path.with_suffix(".lp")

    model.write(str(lp_path))
    lp_path.replace(txt_path)

    print(f"Model exported to {txt_path.resolve()}")


### Model runs and results

In order to be able to analyse the results of our model we use a few functions that store the results into a csv. 

In this section we have 3 functions:

The first function, `export_flows_csv`, saves the active arcs for each period, transport mode and flow

The second function, `export_pipeline_and_emissions_csv`, creates two CSV files:
- Pipeline details: existing pipelines, their installation periods, capacities, flows and utilisation percentages.
- Emissions compliance: captured CO₂, transport emissions, net emissions, the target and whether it is respected in each period. Net emissions are calculated as transport emissions minus captured CO₂.

The third function, `export_costs_and_emissions_csv`, saves operating costs and transport emissions per period, separately for trucks, rail and pipelines. Pipeline investment costs are excluded.

Finally, `export_solution_csvs` calls these three functions and saves the four resulting files in `solution_folder`.

Exports are generated only when the solver has found a feasible solution, including when it reaches the time limit before proving optimality.

In [7]:
def export_flows_csv(model, vars_, filename="solution_flows.csv", tolerance=1e-6):
    if model.SolCount == 0:
        print("No feasible solution to export.")
        return

    rows = []
    for mode in ("truck", "rail", "pipe"):
        for (i, j, t), variable in vars_[f"F_{mode}"].items():
            if variable.X > tolerance:
                rows.append((t, i, j, mode, variable.X))

    rows.sort(key=lambda row: (row[0], row[1], row[2], row[3]))

    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Period", "From", "To", "Transport", "Flow"])
        writer.writerows(rows)

    print(f"Exported {len(rows)} active flows to {filename}")


def export_pipeline_and_emissions_csv(
    model, vars_, params,
    pipeline_filename="solution_pipelines.csv",
    emissions_filename="emissions_compliance.csv",
    tolerance=1e-6,
):
    if model.SolCount == 0:
        print("No feasible solution to export.")
        return

    T = sorted(params["T"])
    N = sorted(params["N"])

    # One row per existing physical pipeline and period.
    # Installation period = first period where w_pipe > 0.5.
    pipeline_rows = []

    for i in N:
        for j in N:
            if i >= j:
                continue

            installation_period = next(
                (t for t in T if vars_["w_pipe"][i, j, t].X > 0.5),
                None,
            )

            if installation_period is None:
                continue

            for t in T:
                if vars_["w_pipe"][i, j, t].X > 0.5:
                    capacity = vars_["S_pipe"][i, j, t].X
                    flow = (
                        vars_["F_pipe"][i, j, t].X
                        + vars_["F_pipe"][j, i, t].X
                    )

                    pipeline_rows.append((
                        t, i, j, installation_period,
                        capacity, flow,
                        100 * flow / capacity if capacity > tolerance else "",
                    ))

    pipeline_rows.sort(key=lambda row: (row[0], row[1], row[2]))

    with open(pipeline_filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow([
            "Period", "Node_1", "Node_2", "Installation_period",
            "Capacity", "Total_flow", "Utilisation_percent",
        ])
        writer.writerows(pipeline_rows)

    # Same emissions expression and target as your model.
    emissions_rows = []

    for t in T:
        captured = sum(
            params["emission_coefficient"]
            * params["E"][i]
            * vars_["c_site"][i, t].X
            for i in params["Nc"]
        )

        transport_emissions = sum(
            params["gamma"][mode]
            * params["Lij"][i, j]
            * vars_[f"F_{mode}"][i, j, t].X
            for mode in ("truck", "rail", "pipe")
            for i in N
            for j in N
        )

        net_emissions = transport_emissions - captured
        target = (
            params["Q_init"]
            + (params["Qt"] - params["Q_init"]) * t / max(T)
        )
        slack = target - net_emissions

        emissions_rows.append((
            t, captured, transport_emissions,
            net_emissions, target, slack,
            slack >= -tolerance,
        ))

    with open(emissions_filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow([
            "Period", "Captured", "Transport_emissions",
            "Net_emissions", "Target", "Slack", "Compliant",
        ])
        writer.writerows(emissions_rows)

    print(f"Exported: {pipeline_filename} and {emissions_filename}")


def export_costs_and_emissions_csv(
    model, vars_, params, filename="costs_and_emissions.csv"
):
    if model.SolCount == 0:
        print("No feasible solution to export.")
        return

    modes = ("truck", "rail", "pipe")
    N = params["N"]

    with open(filename, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow([
            "Period",
            "Truck_operating_cost", "Rail_operating_cost",
            "Pipeline_operating_cost", "Total_operating_cost",
            "Truck_emissions", "Rail_emissions",
            "Pipeline_emissions", "Total_transport_emissions",
            "Captured_CO2", "Net_emissions",
        ])

        for t in sorted(params["T"]):
            costs = {}
            emissions = {}

            for mode in modes:
                costs[mode] = sum(
                    (
                        params["beta"][mode][0] * params["Lij"][i, j]
                        + params["beta"][mode][1]
                    ) * vars_[f"F_{mode}"][i, j, t].X
                    for i in N for j in N
                )

                emissions[mode] = sum(
                    params["gamma"][mode]
                    * params["Lij"][i, j]
                    * vars_[f"F_{mode}"][i, j, t].X
                    for i in N for j in N
                )

            captured = sum(
                params["emission_coefficient"]
                * params["E"][i]
                * vars_["c_site"][i, t].X
                for i in params["Nc"]
            )

            total_emissions = sum(emissions.values())

            writer.writerow([
                t,
                *[costs[mode] for mode in modes],
                sum(costs.values()),
                *[emissions[mode] for mode in modes],
                total_emissions,
                captured,
                total_emissions - captured,
            ])

    print(f"Exported to {filename}")



def export_solution_csvs(model, vars_, params, instance, experiment):
    if model.SolCount == 0:
        print("No feasible solution to export.")
        return

    folder = Path(__file__).resolve().parent / "solution_folder"
    folder.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Hh%M")
    suffix = f"{instance}_{experiment}_{timestamp}.csv"

    export_flows_csv(
        model, vars_,
        filename=folder / f"solution_flows_{suffix}",
    )

    export_pipeline_and_emissions_csv(
        model, vars_, params,
        pipeline_filename=folder / f"solution_pipelines_{suffix}",
        emissions_filename=folder / f"emissions_compliance_{suffix}",
    )

    export_costs_and_emissions_csv(
        model, vars_, params,
        filename=folder / f"costs_and_emissions_{suffix}",
    )


### Solving the model

The following function runs the optimisation for one instance

In [8]:
def run_optimisation_model(model_instance):

    model_builder = OptiModel(
        model_name="small_instance",
        TIME_LIMIT=300,
        model_instance=model_instance
    )

    m, vars_ = model_builder.build()

    m.optimize()
    export_flows_csv(m, vars_)
    export_pipeline_and_emissions_csv(m, vars_, model_builder.params)
    export_costs_and_emissions_csv(m, vars_, model_builder.params, filename="costs_and_emissions.csv")


### Parametre Sensitivity

In order to test the sensitivity of different parametres we run the model multiple times for different values of this parametre.

In [ ]:
def run_experiment(
    experiment,
    instance="medium_instance",
    time_limit=300,
    alpha=None,
    beta_mode=None,
    beta_factor=None,
    reduction=None,
):
    loaders = {
        "small_instance": small_instance,
        "medium_instance": medium_instance,
        "large_instance": large_instance,
    }

    model_builder = OptiModel(
        model_name=f"{instance}_{experiment}",
        TIME_LIMIT=time_limit,
        model_instance=medium_instance
    )
    params = loaders[instance]()
    model_builder.params = params

    metadata = {}

    if alpha is not None:
        params["alpha"] = list(alpha)
        metadata["alpha"] = list(alpha)

    if beta_mode is not None:
        beta = [
            beta_factor * float(value)
            for value in params["beta"][beta_mode]
        ]
        params["beta"][beta_mode] = beta
        metadata.update({
            "transport": beta_mode,
            "factor": beta_factor,
            "beta": beta,
        })

    if reduction is not None:
        params["Q_init"] = 0.0
        params["Qt"] = -reduction * sum(params["E"].values())
        metadata.update({
            "reduction": reduction,
            "Qt": params["Qt"],
        })

    m, vars_ = model_builder.build()

    try:
        m.optimize()

        export_solution_csvs(
            m, vars_, params,
            instance=instance,
            experiment=experiment,
        )

        return {
            "experiment": experiment,
            "instance": instance,
            **metadata,
            "status": m.Status,
            "objective": m.ObjVal if m.SolCount > 0 else None,
            "runtime": m.Runtime,
            "gap": m.MIPGap if m.SolCount > 0 else None,
            "nodes": m.NodeCount,
        }
    finally:
        m.dispose()


def run_multiple_optimisation_runs_for_multiple_beta(
    factors=(0.5, 1.0, 1.5),
    time_limit=300,
    instance="medium_instance",
):
    results = []

    for mode in ("truck", "rail", "pipe"):
        for k, factor in enumerate(factors, start=1):
            results.append(
                run_experiment(
                    experiment=f"beta_{mode}_{k}",
                    instance=instance,
                    time_limit=time_limit,
                    beta_mode=mode,
                    beta_factor=factor,
                )
            )

    return results


def run_multiple_optimisation_runs_for_multiple_alpha(
    alpha_list,
    instance="medium_instance",
    time_limit=300,
):
    return [
        run_experiment(
            experiment=f"alpha_{k}",
            instance=instance,
            time_limit=time_limit,
            alpha=alpha,
        )
        for k, alpha in enumerate(alpha_list, start=1)
    ]


### Plotting results

To see clearer we plot he results saved in a csv for each experiment using the following functions

In [10]:
def plot_solution_csvs(
    costs_csv,
    flows_csv,
    pipelines_csv,
    tolerance=1e-3,
):
    def read_csv(path):
        with open(path, newline="", encoding="utf-8-sig") as file:
            return list(csv.DictReader(file))

    costs = read_csv(costs_csv)
    flows = read_csv(flows_csv)
    pipelines = read_csv(pipelines_csv)

    periods = sorted(int(row["Period"]) for row in costs)
    modes = ("truck", "rail", "pipe")
    colors = {
        "truck": "darkorange",
        "rail": "royalblue",
        "pipe": "seagreen",
    }
    labels = {
        "truck": "Truck",
        "rail": "Rail",
        "pipe": "Pipeline",
    }

    # Total flow carried on arcs, grouped by mode and period.
    flow_totals = {
        mode: {t: 0.0 for t in periods}
        for mode in modes
    }

    for row in flows:
        t = int(row["Period"])
        mode = row["Transport"]
        flow_totals[mode][t] += float(row["Flow"])

    cost_rows = {int(row["Period"]): row for row in costs}

    cost_columns = {
        "truck": "Truck_operating_cost",
        "rail": "Rail_operating_cost",
        "pipe": "Pipeline_operating_cost",
    }

    emissions_columns = {
        "truck": "Truck_emissions",
        "rail": "Rail_emissions",
        "pipe": "Pipeline_emissions",
    }

    # Sets avoid counting a physical pipeline more than once.
    installed = {t: set() for t in periods}
    used = {t: set() for t in periods}
    installation_dates = {}

    for row in pipelines:
        t = int(row["Period"])
        edge = tuple(sorted((
            int(row["Node_1"]),
            int(row["Node_2"]),
        )))

        installed[t].add(edge)
        installation_dates[edge] = int(row["Installation_period"])

        if float(row["Total_flow"]) > tolerance:
            used[t].add(edge)

    newly_built = {
        t: sum(date == t for date in installation_dates.values())
        for t in periods
    }

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    for mode in modes:
        style = {
            "label": labels[mode],
            "color": colors[mode],
            "linewidth": 2,
        }

        axes[0, 0].plot(
            periods,
            [flow_totals[mode][t] for t in periods],
            **style,
        )

        axes[0, 1].plot(
            periods,
            [
                float(cost_rows[t][cost_columns[mode]])
                for t in periods
            ],
            **style,
        )

        axes[1, 0].plot(
            periods,
            [
                float(cost_rows[t][emissions_columns[mode]])
                for t in periods
            ],
            **style,
        )

    axes[0, 0].set_title("Flow by transport mode")
    axes[0, 0].set_ylabel("Total arc flow")

    axes[0, 1].set_title("Operating cost by transport mode")
    axes[0, 1].set_ylabel("Operating cost")

    axes[1, 0].set_title("Transport emissions by mode")
    axes[1, 0].set_ylabel("CO₂ emissions")

    ax = axes[1, 1]
    ax.bar(
        periods,
        [newly_built[t] for t in periods],
        label="Newly built",
        color="lightsteelblue",
    )
    ax.step(
        periods,
        [len(installed[t]) for t in periods],
        where="mid",
        label="Installed",
        color="black",
        linewidth=2,
    )
    ax.step(
        periods,
        [len(used[t]) for t in periods],
        where="mid",
        label="Used",
        color="seagreen",
        linewidth=2,
        linestyle="--",
    )
    ax.set_title("Pipeline deployment and use")
    ax.set_ylabel("Number of pipelines")

    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    for ax in axes.flat:
        ax.set_xlabel("Period")
        ax.grid(alpha=0.3)
        ax.legend()

    fig.tight_layout()
    return fig


def plot_all_experiments(folder="solution_folder"):
    folder = Path(folder)
    figures = []

    for costs_csv in sorted(
        folder.glob("costs_and_emissions_medium_instance_*.csv")
    ):
        suffix = costs_csv.name.removeprefix("costs_and_emissions_")

        flows_csv = folder / f"solution_flows_{suffix}"
        pipelines_csv = folder / f"solution_pipelines_{suffix}"

        if not flows_csv.exists() or not pipelines_csv.exists():
            print(f"Skipped {suffix}: missing flows or pipelines CSV.")
            continue

        fig = plot_solution_csvs(
            costs_csv=costs_csv,
            flows_csv=flows_csv,
            pipelines_csv=pipelines_csv,
        )

        title = Path(suffix).stem
        fig.suptitle(title)
        fig.canvas.manager.set_window_title(title)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        figures.append(fig)

    if figures:
        plt.show()  # Opens all experiment windows together.
    else:
        print(f"No complete experiment results found in {folder.resolve()}.")

    return figures



### Running the model for a given instance

In [ ]:
# run_optimisation_model(small_instance)
run_optimisation_model(medium_instance)
# run_optimisation_model(large_instance)

Set parameter Username
Set parameter LicenseID to value 2880416
Academic license - for non-commercial use only - expires 2027-11-02
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-14700HX, instruction set [SSE2|AVX|AVX2]
Thread count: 20 physical cores, 28 logical processors, using up to 28 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 21987 rows, 15400 columns and 68532 nonzeros (Min)
Model fingerprint: 0x3803c137
Model has 11550 linear objective coefficients
Variable types: 12100 continuous, 3300 integer (3300 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+03]
  Objective range  [5e-04, 4e+04]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e+01, 1e+03]

Presolve removed 15240 rows and 8296 columns
Presolve time: 0.12s
Presolved: 6747 rows, 7104 columns, 27306 nonzeros
Variable types: 5556 continuous, 1548 integer (1548 binary)
Found 

### Running the model for multiple experiments

In [ ]:
alpha_list = [
    [0.00025, 1500],
    [0.0005, 3000],
    [0.00075, 4500],
    [0.001, 6000],
]

results = []
results.extend(
    run_multiple_optimisation_runs_for_multiple_alpha(alpha_list)
)
results.extend(
    run_multiple_optimisation_runs_for_multiple_beta()
)

for result in results:
    print(result)

### Plotting a single experiment

In [ ]:
plot_solution_csvs(
    costs_csv = "old_solutions_1" / "costs_and_emissions_large.csv",
    flows_csv = "old_solutions_1" / "solution_flows_large.csv",
    pipelines_csv = "old_solutions_1" / "solution_pipelines_large.csv",
    tolerance=1e-3,
)

### Plotting Multiple Experiments

In [ ]:
plot_all_experiments(folder="solution_folder")